# AValidate · Audio: BEATs + LoRA vs frozen BEATs

BEATs stays frozen, but each of its 12 attention layers gets small trainable LoRA side paths on the query and value projections (rank 8, about 300k parameters vs BEATs' 90M). Only LoRA and the head train. The head is warm-started from `beats_iter3_v2`, so before training the model **equals frozen BEATs**: the `ep 00` line must show the same validation scores as that run (a wiring check). Same loss, class weights and selection rule as before, so the comparison isolates LoRA.

Needs: GPU runtime (L4 or A100 ideal, T4 works but slower), the `beats_iter3` cache and `beats_iter3_v2` run from notebook 02, and the usual secrets.

In [ ]:
!nvidia-smi -L || echo 'No GPU: switch the runtime to GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
from google.colab import userdata

BRANCH = 'audio/lora'   # switch to 'main' once this branch is merged
REPO = '/content/Audio-Video-Confidence'
token = userdata.get('GH_TOKEN')
url = f'https://{token}@github.com/NUS-AIS-GCs/Audio-Video-Confidence.git'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO], check=True)
else:
    # fetch + hard reset (not pull): always matches GitHub exactly, even if branch history was rewritten
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '-q', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(REPO)
!git log --oneline -1

In [ ]:
!pip -q install -r requirements-audio.txt

In [ ]:
# Weights & Biases: reads WANDB_API_KEY from Colab Secrets (key icon). Skipped if the secret is missing.
WANDB_ENTITY = 'GC2-AValidate'   # team workspace on wandb.ai (shared with the team)
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['WANDB_ENTITY'] = WANDB_ENTITY
    os.environ.setdefault('WANDB_PROJECT', 'avalidate')
    WANDB = '--wandb'
    print('W&B on:', os.environ['WANDB_ENTITY'] + '/' + os.environ['WANDB_PROJECT'])
except Exception as e:
    WANDB = ''
    print('W&B off:', e)

## 1 · Audio
LoRA reads the waveforms directly (BEATs' output changes as it learns, so cached embeddings can't be used). About 9 minutes for a fresh session.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.data_ave

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.extract_audio

## 2 · Sanity check
LoRA + head from scratch on 32 clips: **val sec-acc should reach about 0.95+**.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.train --encoder iter3 --lora-r 8 --overfit 32 --epochs 20 --lr 1e-3 --run-id overfit_lora

## 3 · Train BEATs + LoRA (rank 8)
10 epochs, batch 16, learning rate 2e-4, warm start from `beats_iter3_v2`. Check the `ep 00` line matches `beats_iter3_v2`'s validation scores (sec-F1 ≈ 0.669, clip-acc ≈ 0.786). Each epoch runs the full transformer, so expect several minutes per epoch on a T4 and much less on an L4/A100. If LoRA never beats the warm start on validation, the saved model is the warm start itself.

In [ ]:
%cd /content/Audio-Video-Confidence
RUN_ID = 'beats_lora_r8_v1'
WANDB = globals().get('WANDB', '')
if not WANDB: print('W&B is OFF: run the W&B cell first if you want tracking')
!python -m audio.train --run-id {RUN_ID} --encoder iter3 --lora-r 8 --init-head beats_iter3_v2 --epochs 10 --batch 16 --lr 2e-4 {WANDB}

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.evaluate --run-id {RUN_ID} --batch 32 {WANDB}
from IPython.display import Image, display
display(Image(f'/content/drive/MyDrive/AValidate/artifacts/audio_baseline/{RUN_ID}/confusion_test.png'))

## 4 · Side by side

In [ ]:
import json, pandas as pd
rows = {}
for run in ['cnn_v4', 'beats_iter3_v2', 'beats_lora_r8_v1', 'beats_lora_r4_v1', 'beats_lora_r16_v1']:
    p = f'/content/drive/MyDrive/AValidate/artifacts/audio_baseline/{run}/results.json'
    if os.path.exists(p):
        rows[run] = json.load(open(p))['test']
pd.DataFrame(rows).T.drop(columns='n_clips').round(3)

## 5 · Optional: rank comparison
Only if time allows: same run with rank 4 and rank 16, to see whether LoRA's size matters. Re-run the table above afterwards.

In [ ]:
%cd /content/Audio-Video-Confidence
# for r in [4, 16]:
#     !python -m audio.train --run-id beats_lora_r{r}_v1 --encoder iter3 --lora-r {r} --lora-alpha {2*r} --init-head beats_iter3_v2 --epochs 10 --batch 16 --lr 2e-4 {WANDB}
#     !python -m audio.evaluate --run-id beats_lora_r{r}_v1 --batch 32 {WANDB}

## 6 · Export for the team (only if LoRA beats frozen BEATs on validation)
Writes `embeddings/audio_beats_lora_r8_v1/` with `{split}_emb.npy` (N, 10, 256), `{split}_probs.npy` (N, 10, 29), `{split}_ids.csv`.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.export_embeddings --run-id beats_lora_r8_v1